# Figure 1. Study design and summary of glycoproteomic profiling

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
# library(treemapify)
library(RColorBrewer)
library(tidyverse)
library(data.table)

## Fig 1b

In [ ]:
venn_list <- readRDS('data/Figure_1b.rds')

venn_counts <- c(
  glycation_only = length(setdiff(venn_list[["Glycation"]], venn_list[["Glycosylation"]])),
  overlap = length(intersect(venn_list[["Glycation"]], venn_list[["Glycosylation"]])),
  glycosylation_only = length(setdiff(venn_list[["Glycosylation"]], venn_list[["Glycation"]]))
)
venn_pct <- scales::percent(venn_counts / sum(venn_counts), accuracy = 0.1)

circle_df <- bind_rows(
  data.frame(group = "Glycation", angle = seq(0, 2 * pi, length.out = 361), cx = 0, cy = 0, r = 1),
  data.frame(group = "Glycosylation", angle = seq(0, 2 * pi, length.out = 361), cx = 1.55, cy = 0, r = 1)
) %>%
  mutate(x = cx + r * cos(angle), y = cy + r * sin(angle))

f1b <- ggplot(circle_df, aes(x, y, group = group, fill = group)) +
  geom_polygon(alpha = 0.62, color = "grey25", linewidth = 0.4) +
  annotate("text", x = -0.42, y = 0.12, label = scales::comma(venn_counts[["glycation_only"]]), size = 4.8) +
  annotate("text", x = -0.42, y = -0.12, label = paste0("(", venn_pct[["glycation_only"]], ")"), size = 3.4, color = "grey30") +
  annotate("text", x = 0.78, y = 0.12, label = scales::comma(venn_counts[["overlap"]]), size = 4.6) +
  annotate("text", x = 0.78, y = -0.12, label = paste0("(", venn_pct[["overlap"]], ")"), size = 3.2, color = "grey30") +
  annotate("text", x = 1.98, y = 0.12, label = scales::comma(venn_counts[["glycosylation_only"]]), size = 4.8) +
  annotate("text", x = 1.98, y = -0.12, label = paste0("(", venn_pct[["glycosylation_only"]], ")"), size = 3.4, color = "grey30") +
  annotate("text", x = -0.35, y = 1.30, label = "Glycation", size = 4.5, fontface = "bold") +
  annotate("text", x = 1.95, y = 1.30, label = "Glycosylation", size = 4.5, fontface = "bold") +
  scale_fill_manual(values = c("Glycation" = "#C44E52", "Glycosylation" = "#4C72B0")) +
  coord_fixed(xlim = c(-1.03, 2.65), ylim = c(-1.02, 1.42), expand = FALSE, clip = "off") +
  theme_void(base_size = 14) +
  theme(legend.position = "none", plot.margin = margin(t = 1, r = 2, b = 1, l = 2))


## Fig 1C

In [ ]:
glycan_class_summary <- readRDS('data/Figure_1d.rds') %>%
  mutate(
    label_y = rev(cumsum(rev(n)) - 0.5 * rev(n)),
    label_radius = case_when(
      GlycanClass == "Other" ~ 1.25,
      GlycanClass == "Paucimannose" ~ 0.76,
      TRUE ~ 1.0
    )
  )
# glycan-class palette shared across Fig 1c/2c/3
cus_colors <- c(
  "Paucimannose" = "#A6CEE3",
  "Hybrid/Complex" = "#1F78B4",
  "High-mannose" = "#66C2A5",
  "Glycation" = "#C44E52",
  "Other" = "grey50"
)
f1c <- ggplot(glycan_class_summary, aes(x = 1, y = n, fill = GlycanClass)) +
  geom_col(width = 1, color = "white") +
  scale_x_continuous(limits = c(0.50, 1.50), expand = expansion(add = 0)) +
  coord_polar(theta = "y") +
  geom_text(aes(x = label_radius, y = label_y, label = Label), inherit.aes = FALSE, size = 4.5) +
  theme_void(base_size = 14) +
  scale_fill_manual(values = cus_colors, labels = c("Hybrid/Complex" = "Hybrid/complex")) +
  labs(fill = "Glycan class") +
  theme(legend.position = "bottom")
f1c


## Fig 1D

In [ ]:
library(patchwork)

counts_per_sample <- readRDS('data/Figure_1c.rds')

feature_order <- c('Glycans', 'Glycoproteins', 'Glycosites', 'Glycanpeptides', 'Glycopeptiforms')
feature_labels <- c('Glycans' = 'Glycan compositions', 'Glycoproteins' = 'Glycoproteins',
                    'Glycosites' = 'Glycosites', 'Glycanpeptides' = 'Glycopeptides',
                    'Glycopeptiforms' = 'Glycopeptiforms')
counts_per_sample <- counts_per_sample %>%
  mutate(Feature = factor(Feature, levels = feature_order))

d_fill <- c("Glycation" = "#C44E52", "Glycosylation" = "#4C72B0")
d_theme <- theme_classic(base_size = 12) +
  theme(
    legend.position = "none",
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.75, "lines"),
    panel.grid = element_blank(),
    plot.title = element_text(face = "bold", size = 8.5, hjust = 0.5, margin = margin(b = 6)),
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    axis.text.y = element_text(size = 8.2),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks.y = element_line(color = "black", linewidth = 0.35),
    axis.title.y = element_text(size = 9.5, margin = margin(r = 3)),
    plot.margin = margin(2, 5, 2, 5)
  )

broken_y_trans <- function(lower = 1800, upper = 6000, compressed_gap = 180) {
  removed <- upper - lower - compressed_gap
  scales::trans_new(
    name = "broken_y",
    transform = function(x) ifelse(
      x <= lower, x,
      ifelse(x >= upper, x - removed, lower + (x - lower) * compressed_gap / (upper - lower))
    ),
    inverse = function(x) ifelse(
      x <= lower, x,
      ifelse(x >= lower + compressed_gap, x + removed,
             lower + (x - lower) * (upper - lower) / compressed_gap)
    )
  )
}

make_d_panel <- function(feature, y_limits, show_y_title = FALSE, broken_axis = FALSE) {
  p <- counts_per_sample %>%
    filter(Feature == feature) %>%
    mutate(
      x_pos = if (feature == "Glycans") 1.5 else
        match(GlycanTypeSummary, c("Glycation", "Glycosylation"))
    ) %>%
    ggplot(aes(x = x_pos, y = Count, fill = GlycanTypeSummary, group = GlycanTypeSummary)) +
    geom_boxplot(width = if (feature == "Glycans") 0.74 else 0.58,
                 outlier.shape = 16, outlier.size = 0.5, linewidth = 0.35) +
    scale_x_continuous(breaks = NULL, expand = expansion(add = 0)) +
    coord_cartesian(xlim = c(0.45, 2.55), clip = "off") +
    scale_fill_manual(values = d_fill, breaks = c("Glycation", "Glycosylation"), drop = FALSE) +
    labs(
      title = unname(feature_labels[[feature]]), x = NULL,
      y = if (show_y_title) "Detected per sample" else NULL, fill = "Glycan type"
    ) +
    d_theme
  if (broken_axis) {
    p <- p +
      scale_y_continuous(
        trans = broken_y_trans(), limits = y_limits,
        breaks = c(1200, 1400, 1600, 1800, 6000, 6500, 7000, 7500),
        labels = scales::label_number(big.mark = ","),
        expand = expansion(mult = c(0, 0.03))
      ) +
      annotate("rect", xmin = 0.35, xmax = 0.55, ymin = 3000, ymax = 4800,
               fill = "white", color = NA) +
      annotate("segment", x = 0.38, xend = 0.52, y = 3100, yend = 3600,
               linewidth = 0.35, color = "black") +
      annotate("segment", x = 0.38, xend = 0.52, y = 4100, yend = 4600,
               linewidth = 0.35, color = "black")
  } else {
    p <- p + scale_y_continuous(
      limits = y_limits, breaks = scales::pretty_breaks(n = 3),
      labels = scales::label_number(big.mark = ","),
      expand = expansion(mult = c(0, 0.05))
    )
  }
  p
}

f1d_compositions <- make_d_panel("Glycans", c(220, 240), TRUE)  # updated 2026-08-07: range was 245-290, now 225-234 post w54-resolve + Method2 fix (see Figure1_data_preparation.ipynb)
f1d_proteins <- make_d_panel("Glycoproteins", c(450, 780))
f1d_sites <- make_d_panel("Glycosites", c(1100, 1650))
f1d_peptides <- make_d_panel("Glycanpeptides", c(1100, 1660))
f1d_peptiforms <- make_d_panel("Glycopeptiforms", c(1100, 7500), broken_axis = TRUE)

d_legend <- cowplot::get_legend(
  f1d_proteins + theme(legend.position = "bottom") +
    guides(fill = guide_legend(nrow = 1, title = NULL))
)
f1d_plots <- (f1d_compositions | f1d_proteins | f1d_sites | f1d_peptides | f1d_peptiforms) +
  plot_layout(widths = c(0.78, 1, 1, 1, 1.18))
f1d <- f1d_plots / wrap_elements(full = d_legend) +
  plot_layout(heights = c(1, 0.09))


## combine panels

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)
img <- image_read("data/glycoQTL_study_design.png")

crop_plot_panel <- function(plot, width, height, dpi = 600, fuzz = 8, scale = 1,
                            valign = 0.5, halign = 0.5, x_offset = 0) {
  raw_png <- tempfile(fileext = ".png")
  trim_png <- tempfile(fileext = ".png")
  ggsave(raw_png, plot, width = width, height = height, dpi = dpi, bg = "white")
  image_trim(image_read(raw_png), fuzz = fuzz) %>%
    image_write(trim_png)
  ggdraw() +
    draw_image(trim_png, x = x_offset, scale = scale, valign = valign, halign = halign) +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))
}

f1a <- ggdraw() +
  draw_image(img, scale = 1) +
  theme(plot.margin = margin(10, 4, 0, 4))

# b is saved, trimmed, and loaded back as a raster panel, matching the c workflow
f1b_panel <- crop_plot_panel(
  f1b,
  width = 3.25, height = 2.25, fuzz = 10, scale = 1.04,
  valign = 1.0, halign = 0.5, x_offset = 0
) +
  theme(plot.margin = margin(t = -8, r = 0, b = 0, l = 0))

f1c_pie_panel <- crop_plot_panel(
  f1c +
    theme(legend.position = "none", plot.margin = margin(0, 0, 0, 0)) +
    guides(fill = "none"),
  width = 3.05, height = 3.05, fuzz = 10, scale = 0.8
) +
  theme(plot.margin = margin(-10, 0, 0, 0))

f1c_legend <- get_legend(
  f1c +
    theme_void(base_size = 14) +
    theme(
      legend.position = "bottom",
      legend.key.size = unit(0.78, "lines"),
      legend.text = element_text(size = 10),
      legend.title = element_blank(),
      legend.margin = margin(-10, 0, 0, 0),
      legend.box.margin = margin(-10, 0, 0, 0)
    ) +
    guides(fill = guide_legend(nrow = 2, byrow = TRUE))
)

f1c_panel <- ggdraw() +
  draw_plot(f1c_pie_panel, x = 0, y = 0.18, width = 1, height = 0.78) +
  draw_grob(f1c_legend, x = 0, y = 0.00, width = 1, height = 0.23) +
  theme(plot.margin = margin(0, 0, 0, 0))

f1d <- f1d &
  theme(plot.margin = margin(t = 18, r = 5, b = 6, l = 3))

f1d_panel <- wrap_elements(full = f1d)

left_col   <- f1b_panel / f1c_panel + plot_layout(heights = c(0.58, 1.20))
bottom_row <- (left_col | f1d_panel) + plot_layout(widths = c(0.68, 1.62))

combined_fig <- f1a / bottom_row +
  plot_layout(heights = c(1.15, 0.975))

final_figure <- (combined_fig + plot_annotation(tag_levels = "a")) &
  theme(plot.tag = element_text(face = "bold", size = 15))

ggsave("Figure1.pdf", final_figure, width = 11.8, height = 8.1, dpi = 300)
